In [ ]:
# Repository and Imports

%matplotlib inline

from pathlib import Path
from time import perf_counter
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

from dataclasses import replace

repository_root = Path.cwd().resolve()

if not (repository_root / "ramsey").is_dir():
    parent_candidate = repository_root.parent

    if (parent_candidate / "ramsey").is_dir():
        repository_root = parent_candidate
    else:
        raise RuntimeError(
            "Could not locate the refactor repository root."
        )

if str(repository_root) not in sys.path:
    sys.path.insert(
        0,
        str(repository_root),
    )


from ramsey import (
    RDangerObjective,
    REnvironment,
    REnvironmentConfig,
    RGraph,
    RMonochromaticObjective,
    RProblem,
    RRandomConstruction,
    RSQLiteArchive,
    RTabuMemory,
    RTabuMemoryConfig,
)
from ramsey.RPlot import (
    plot_coloring_histogram,
)
from ramsey.nn import (
    RCheckpointSchedule,
    RModelConfig,
    RPPOConfig,
    RPPOTrainer,
    RPairPolicyValueNetwork,
    RRolloutConfig,
    RRolloutReward,
    RTrainingConfig,
    create_optimizer,
    load_training_checkpoint,
)

In [ ]:
# Experiment Configuration

RANDOM_SEED = 20260804

N_VERTICES = 43
RUN_NAME = "ppo-k43"

TRAINING_ITERATIONS = 50
CHECKPOINT_INTERVAL = 10

ROLLOUT_STEPS = 256
EDGE_TABU_TENURE = 20
VISITED_STATE_WINDOW = 2_000

USE_DANGER_REWARD = False
DANGER_DECAY = 0.25

DATABASE_PATH = (
    repository_root
    / "data"
    / "ramsey_colorings.sqlite3"
)

CHECKPOINT_DIRECTORY = (
    repository_root
    / "checkpoints"
)

# Use None to begin with a new network.
#
# Example:
#
# RESUME_CHECKPOINT = (
#     CHECKPOINT_DIRECTORY
#     / "ramsey_policy_iteration_000100.pt"
# )
RESUME_CHECKPOINT: Path | None = CHECKPOINT_DIRECTORY / "ramsey_policy_iteration_000049.pt"

# When resuming, set this to override the saved optimizer learning
# rate. Leave it as None to use the checkpoint's learning rate.
NEW_LEARNING_RATE: float | None = 1.0e-4

In [ ]:
# Randomness and CUDA Device 

rng = np.random.default_rng(
    RANDOM_SEED
)

torch.manual_seed(
    RANDOM_SEED
)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(
        RANDOM_SEED
    )

if torch.cuda.is_available():
    device = torch.device(
        "cuda",
        torch.cuda.current_device(),
    )
else:
    device = torch.device(
        "cpu"
    )

print(
    "Repository root:",
    repository_root,
)

print(
    "Device:",
    device,
)

In [ ]:
# Ramsey Problem and Graph Scaffolding

graph_start = perf_counter()

problem = RProblem.r55(
    n_vertices=N_VERTICES,
)

graph = RGraph(
    problem
)

graph_elapsed = (
    perf_counter()
    - graph_start
)

print(
    "Problem:",
    problem,
)

print(
    "Edges:",
    f"{graph.number_of_edges:,}",
)

print(
    "Graph construction:",
    f"{graph_elapsed:.3f} seconds",
)

In [ ]:
# Fresh Model Configuration (used when RESUME_CHECKPOINT is False)
model_config = RModelConfig(
    input_size=3,
    hidden_size=64,
    number_of_layers=4,
    dropout=0.0,
)

rollout_config = RRolloutConfig(
    rollout_steps=ROLLOUT_STEPS,
    discount=0.995,
    gae_lambda=0.95,
    reward_scale=10.0,
    reward_source=(
        RRolloutReward.OBJECTIVE
        if USE_DANGER_REWARD
        else RRolloutReward.EXACT_SCORE
    ),
    normalize_advantages=True,
)

ppo_config = RPPOConfig(
    update_epochs=4,
    minibatch_size=16,
    clip_ratio=0.20,
    value_loss_weight=0.00,
    entropy_weight=0.0,
    maximum_gradient_norm=0.50,
    learning_rate=5.0e-4,
    target_kl=None,
)

In [ ]:
# Model Creation or Restoration

if RESUME_CHECKPOINT is None:
    network = RPairPolicyValueNetwork(
        graph,
        model_config,
    ).to(device)

    optimizer = create_optimizer(
        network,
        ppo_config,
    )

    start_iteration = 0

    print(
        "Created a new network."
    )

else:
    restored = load_training_checkpoint(
        RESUME_CHECKPOINT,
        graph=graph,
        device=device,
        rng=rng,
        new_learning_rate=NEW_LEARNING_RATE,
    )
    ppo_config = replace(
    ppo_config,
    target_kl=0.02,
)
    network = restored.network
    optimizer = restored.optimizer
    model_config = restored.model_config
    rollout_config = restored.rollout_config
    ppo_config = restored.ppo_config

    start_iteration = (
        restored.completed_iteration
        + 1
    )

    print(
        "Restored checkpoint:",
        RESUME_CHECKPOINT,
    )

    print(
        "Completed iteration:",
        restored.completed_iteration,
    )

print(
    "Starting iteration:",
    start_iteration,
)

print(
    "Trainable parameters:",
    f"{network.trainable_parameter_count:,}",
)

print(
    "Learning rate:",
    optimizer.param_groups[0]["lr"],
)

print(
    "Rollout steps:",
    rollout_config.rollout_steps,
)

In [ ]:
# Construction, Environment, and Archive

construction = RRandomConstruction(
    rng
)

if USE_DANGER_REWARD:
    objective = RDangerObjective(
        decay=DANGER_DECAY,
    )
else:
    objective = RMonochromaticObjective()

memory = RTabuMemory(
    number_of_edges=graph.number_of_edges,
    config=RTabuMemoryConfig(
        edge_tenure=EDGE_TABU_TENURE,
        visited_state_window=(
            VISITED_STATE_WINDOW
        ),
    ),
)

environment = REnvironment(
    graph=graph,
    objective=objective,
    memory=memory,
    config=REnvironmentConfig(
        max_steps=rollout_config.rollout_steps,
        use_aspiration=True,
    ),
)

# Close an earlier notebook connection if this cell is rerun.
existing_archive = globals().get(
    "archive"
)

if existing_archive is not None:
    existing_archive.close()

archive = RSQLiteArchive(
    DATABASE_PATH
)

print(
    "Construction:",
    construction.name,
)

print(
    "Objective:",
    objective.name,
)

print(
    "Archived colorings:",
    archive.coloring_count(
        graph
    ),
)

print(
    "Archive best:",
    archive.best_score(
        graph
    ),
)

In [ ]:
# Assemble the Trainer

trainer = RPPOTrainer(
    graph=graph,
    construction=construction,
    environment=environment,
    network=network,
    optimizer=optimizer,
    device=device,
    rng=rng,
    rollout_config=rollout_config,
    ppo_config=ppo_config,
    archive=archive,
)

checkpoint_schedule = RCheckpointSchedule(
    directory=CHECKPOINT_DIRECTORY,
    interval=CHECKPOINT_INTERVAL,
    save_final=True,
)

training_config = RTrainingConfig(
    run_name=RUN_NAME,
    iterations=TRAINING_ITERATIONS,
    start_iteration=start_iteration,
    stop_on_solution=True,
)

print(
    "Trainer:",
    type(trainer).__name__,
)

print(
    "Iterations:",
    training_config.iterations,
)

print(
    "Starting iteration:",
    training_config.start_iteration,
)

In [ ]:
# Progress Reporting

def report_training_iteration(
    iteration_result,
) -> None:
    metrics = iteration_result.metrics

    if metrics is None:
        metric_text = "no parameter update"
    else:
        metric_text = (
            f"policy={metrics.policy_loss:+.4f} | "
            f"value={metrics.value_loss:.4f} | "
            f"entropy={metrics.entropy:.4f} | "
            f"KL={metrics.approximate_kl:.6f} | "
            f"clipped={metrics.clipped_fraction:.4f}"
        )

    checkpoint_text = (
        f" | checkpoint={iteration_result.checkpoint_path.name}"
        if iteration_result.checkpoint_path is not None
        else ""
    )

    archive_text = ""

    if iteration_result.archive_record is not None:
        archive_record = (
            iteration_result.archive_record
        )

        archive_text = (
            f" | archive_id={archive_record.coloring_id} "
            f"archive_score={archive_record.score}"
        )

        if iteration_result.new_archive_best:
            archive_text += " NEW DATABASE BEST"

    print(
        f"Iteration {iteration_result.iteration:6d} | "
        f"initial={iteration_result.initial_score:5d} | "
        f"final={iteration_result.final_score:5d} | "
        f"best={iteration_result.best_score:5d} | "
        f"reward={iteration_result.total_scaled_reward:+9.3f} | "
        f"{metric_text}"
        f"{archive_text}"
        f"{checkpoint_text}"
    )

In [ ]:
# Run Training 

training_start = perf_counter()

training_result = trainer.run(
    training_config,
    checkpoint_schedule=checkpoint_schedule,
    observer=report_training_iteration,
)

training_elapsed = (
    perf_counter()
    - training_start
)

print()

print(
    "Completed iterations:",
    training_result.completed_iterations,
)

print(
    "Best score:",
    training_result.best_score,
)

print(
    "Solved:",
    training_result.solved,
)

print(
    "Total time:",
    f"{training_elapsed:.3f} seconds",
)

print(
    "Mean time per iteration:",
    (
        f"{training_elapsed / training_result.completed_iterations:.3f} "
        "seconds"
    ),
)

print(
    "Value loss weight:",
    ppo_config.value_loss_weight,
)

In [ ]:
# Plot Training Scores

iteration_numbers = np.asarray(
    [
        item.iteration
        for item
        in training_result.iteration_results
    ],
    dtype=np.int64,
)

initial_scores = np.asarray(
    [
        item.initial_score
        for item
        in training_result.iteration_results
    ],
    dtype=np.int64,
)

final_scores = np.asarray(
    [
        item.final_score
        for item
        in training_result.iteration_results
    ],
    dtype=np.int64,
)

best_scores = np.asarray(
    [
        item.best_score
        for item
        in training_result.iteration_results
    ],
    dtype=np.int64,
)

running_best_scores = np.minimum.accumulate(
    best_scores
)

figure, axis = plt.subplots(
    figsize=(11, 5.5),
    constrained_layout=True,
)

axis.plot(
    iteration_numbers,
    initial_scores,
    label="initial",
    alpha=0.55,
)

axis.plot(
    iteration_numbers,
    final_scores,
    label="final",
    alpha=0.75,
)

axis.plot(
    iteration_numbers,
    best_scores,
    label="rollout best",
    alpha=0.85,
)

axis.plot(
    iteration_numbers,
    running_best_scores,
    label="run best",
    linewidth=2.5,
)

axis.set_xlabel(
    "Training iteration"
)

axis.set_ylabel(
    "Monochromatic K5 score"
)

axis.set_title(
    RUN_NAME
)

axis.grid(
    linestyle="--",
    alpha=0.3,
)

axis.legend()

plt.show()

In [ ]:
# Plot the Histogram for the Best Coloring

best_iteration = (
    training_result.best_iteration
)

print(
    "Best iteration:",
    best_iteration.iteration,
)

print(
    "Initial score:",
    best_iteration.initial_score,
)

print(
    "Final score:",
    best_iteration.final_score,
)

print(
    "Best score:",
    best_iteration.best_score,
)

plot_coloring_histogram(
    best_iteration.best_coloring,
    title=(
        f"Best training coloring — "
        f"iteration {best_iteration.iteration:,} — "
        f"score {best_iteration.best_score:,}"
    ),
)

plt.show()

In [ ]:
print(
    "Database:",
    DATABASE_PATH.resolve(),
)

print(
    "Stored K43 colorings:",
    archive.coloring_count(
        graph
    ),
)

print(
    "Database best:",
    archive.best_score(
        graph
    ),
)

for record in archive.best_colorings(
    limit=10,
    graph=graph,
):
    print(
        f"ID={record.coloring_id:6d} | "
        f"score={record.score:5d} | "
        f"run={record.run_name} | "
        f"iteration={record.iteration:6d} | "
        f"seen={record.times_seen}"
    )